# 09 — Guardrails and human approval
Validate before the Crew runs. Check the report before accepting it. A mock
email is a side effect, so it requires approval immediately before execution.



## Problem

An agent's answer must not authorize its own customer-facing side effect.


## Concept

Validation and human approval are executable boundaries outside the model.

Guardrails are executable checks at input, output, and action boundaries. A human approval decision is external state; no prompt can manufacture it.


## Abstraction

request → validation → report check → approval gate → mock action. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
input ──▶ validate_question ──▶ Crew report
                                      │
                               validate_report
                                      │
                           approval? ──no──▶ blocked
                               │ yes
                               ▼
                         idempotent mock email
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Reject an invalid question and validate a deliberately sourced report

Reject an invalid question and validate a deliberately sourced report.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.control import mock_email, validate_question, validate_report
from workshop.schemas import DecisionReport, Evidence

try:
    validate_question("hi")
except ValueError as error:
    print("Input blocked:", error)

report = DecisionReport(
    recommendation="pilot",
    rationale="A private deployment is possible, but deletion and throughput need verification.",
    evidence=[
        Evidence(source_id="architecture.md", claim="Private VPC deployment is possible."),
        Evidence(source_id="security.md", claim="Data deletion guarantees are unverified."),
    ],
    risks=["Unverified deletion guarantees"],
    next_steps=["Run a bounded pilot"],
)
print("Output check:", validate_report(report))



## Exercise the approval boundary
First run is denied. Second run records a local mock action. A repeated
action ID is ignored, which matters when an interrupted Flow resumes.



## Step 2 — Exercise denied, approved, and replayed mock actions using the same ID

Exercise denied, approved, and replayed mock actions using the same ID.


In [ ]:
print(mock_email(report, approved=False, action_id="lesson-09-demo"))
print(mock_email(report, approved=True, action_id="lesson-09-demo"))
print(mock_email(report, approved=True, action_id="lesson-09-demo"))


## Execution flow

request → validation → report check → approval gate → mock action. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

Never treat an agent's request as human approval. An action must pass validation and an external approval flag; retries must reuse an idempotency key.


## Production implication

Authorize side effects through the host application, store approval evidence, and use idempotency keys. Model output is never an authorization token.


## Your turn

Run the denied, approved, and repeated-action cases in order.

**Checkpoint:** Denied writes nothing; approved writes one local record; repeat writes no second record.
